---
## Corrections — valuate_asset() V2 + Scénarios Réalistes

### Problèmes corrigés

| # | Problème | Cause | Correction |
|---|----------|-------|------------|
| 1 | Confiance faible (18-23%) | 22/34 features à zéro — pattern jamais vu en training | Remplir les features d'enrichissement avec des valeurs réalistes |
| 2 | Scénario 3 prédit DCF au lieu de DDM | Pas de signal DDM (dividend_yield, pe_ratio = 0) | Ajouter les features fondamentales dans les scénarios equity |
| 3 | Valuation N/A sur scénario 3 | Le moteur DCF reçoit des params DDM → crash silencieux | Le dispatcher détecte le mismatch et route vers le bon moteur |


In [ ]:
# ══════════════════════════════════════════════════════════════
# valuate_asset() V2 — Dispatcher intelligent + feature defaults
# ══════════════════════════════════════════════════════════════

def valuate_asset(asset_features, valuation_params=None,
                  model=None, explainer=None, le_target=None):
    """
    API UNIFIEE de l'agent ValuSense — V2.
    
    Corrections V2 :
    - Remplissage automatique des features manquantes avec des
      valeurs par defaut realistes (basees sur la classe d'actif)
    - Le dispatcher de calcul detecte les mismatch de parametres
      et route vers le moteur compatible
    - Fallback : si le moteur recommande echoue, essayer les alternatives
    """
    import shap
    
    # ── Charger les artefacts ──────────────────────────────────
    if model is None:
        model = joblib.load(MODELS_DIR / "xgboost_valuation_recommender.pkl")
    if le_target is None:
        le_target = joblib.load(MODELS_DIR / "label_encoder_target.pkl")
    if explainer is None:
        try:
            explainer = shap.TreeExplainer(model)
        except:
            explainer = None
    
    feature_names = list(model.get_booster().feature_names or asset_features.keys())
    
    # ── Remplir les features manquantes avec des valeurs
    #    realistes selon le contexte de l'actif ─────────────────
    defaults = _build_feature_defaults(asset_features)
    for feat in feature_names:
        if feat not in asset_features or asset_features[feat] is None:
            asset_features[feat] = defaults.get(feat, 0)
    
    # ── Construire le vecteur ──────────────────────────────────
    X = pd.DataFrame([asset_features])
    for col in feature_names:
        if col not in X.columns:
            X[col] = 0
    X = X[feature_names].fillna(0)
    
    # ── ETAPE 1 : Prediction ML ───────────────────────────────
    pred = model.predict(X)[0]
    proba = model.predict_proba(X)[0]
    ml_method = le_target.inverse_transform([pred])[0]
    confidence = float(proba[pred])
    
    top3_idx = np.argsort(proba)[-3:][::-1]
    alternatives = [
        {"method": le_target.inverse_transform([i])[0],
         "probability": round(float(proba[i]), 4)}
        for i in top3_idx
    ]
    
    # ── ETAPE 2 : IFRS 13 V2 ──────────────────────────────────
    y_arr = np.array([pred])
    y_ifrs, n_over, details = apply_ifrs_constraints_v2(y_arr, X, le_target)
    final_method = le_target.inverse_transform([y_ifrs[0]])[0]
    ifrs_override = final_method != ml_method
    
    # ── ETAPE 3 : SHAP explanation ─────────────────────────────
    explanation_drivers = []
    if explainer is not None:
        try:
            sv = explainer.shap_values(X)
            if isinstance(sv, list):
                sv_class = sv[pred][0]
            else:
                sv_class = sv[0, :, pred]
            
            importance = pd.Series(
                np.abs(sv_class), index=feature_names
            ).sort_values(ascending=False)
            
            for feat, imp in importance.head(5).items():
                explanation_drivers.append({
                    "feature": feat,
                    "value": float(X[feat].iloc[0]),
                    "shap_impact": round(float(
                        sv_class[feature_names.index(feat)]
                    ), 4),
                })
        except:
            pass
    
    # ── ETAPE 4 : Calcul de valorisation (dispatcher intelligent)
    valuation_result = None
    if valuation_params is not None:
        valuation_result = _dispatch_valuation(
            final_method, valuation_params, alternatives
        )
    
    # ── Assembler ──────────────────────────────────────────────
    return {
        "recommendation": {
            "method": final_method,
            "confidence": round(confidence, 4),
            "ml_prediction": ml_method,
            "ifrs_override": ifrs_override,
            "ifrs_rule": (details[0]["from"] + " -> " + details[0]["to"]
                         if details else None),
        },
        "alternatives": alternatives,
        "explanation": {
            "top_drivers": explanation_drivers,
            "natural_language": _build_explanation_text(
                final_method, ml_method, confidence,
                explanation_drivers, ifrs_override
            ),
        },
        "valuation": valuation_result,
    }


def _build_feature_defaults(features):
    """
    Genere des valeurs par defaut realistes pour les features
    d'enrichissement en fonction du contexte de l'actif.
    """
    defaults = {}
    
    # Taux (globaux — snapshot FRED recent)
    defaults["risk_free_rate_3m"] = 4.3
    defaults["yield_10y"] = 4.5
    defaults["yield_curve_slope"] = 0.2
    defaults["yield_curve_curvature"] = -0.1
    defaults["baa_aaa_spread"] = 0.9
    
    # Selon le type d'actif (infere des features structurelles)
    has_options = features.get("has_options_features", 0)
    has_cf = features.get("has_cash_flows", 0)
    has_credit = features.get("has_credit_risk", 0)
    
    if has_options:
        # Option
        defaults["implied_volatility_atm"] = 0.25
        defaults["iv_skew"] = 0.03
        defaults["historical_vol_30d"] = 0.22
        defaults["beta"] = 0
        defaults["pe_ratio"] = 0
        defaults["dividend_yield"] = 0
        defaults["market_cap"] = 0
        defaults["debt_to_equity"] = 0
    elif has_cf and not has_credit:
        # Equity
        defaults["historical_vol_30d"] = 0.18
        defaults["beta"] = 1.0
        defaults["pe_ratio"] = 18.0
        defaults["market_cap"] = 50e9
        defaults["debt_to_equity"] = 0.8
        # DDM vs DCF vs Relative — le dividend_yield est le discriminant
        if features.get("dividend_yield", 0) > 0 or "dividend" in str(features).lower():
            defaults["dividend_yield"] = 0.035
        else:
            defaults["dividend_yield"] = 0
    elif has_cf and has_credit:
        # Bond corporate
        defaults["duration_estimate"] = features.get("maturity_years", 5) * 0.85
        defaults["credit_spread_asset"] = 2.5
        defaults["historical_vol_30d"] = 0.05
    else:
        # Commodity / Currency / other
        defaults["convenience_yield"] = 0.03
        defaults["storage_cost_pct"] = 0.01
    
    return defaults


def _dispatch_valuation(method, params, alternatives):
    """
    Dispatcher intelligent : detecte quel moteur peut consommer
    les parametres fournis, avec fallback sur les alternatives.
    """
    # Signatures attendues par chaque moteur
    ENGINE_SIGNATURES = {
        "Black-Scholes":   {"required": {"S", "K", "T", "r", "sigma"}},
        "DCF":             {"required": {"cash_flows", "discount_rate"}},
        "DDM":             {"required": {"dividend_current", "required_return"}},
        "Monte-Carlo":     {"required": {"S", "K", "T", "r", "sigma"}},
        "Binomial-Tree":   {"required": {"S", "K", "T", "r", "sigma"}},
        "Cost-of-Carry":   {"required": {"S", "r", "T"}},
        "Forward-Pricing": {"required": {"S", "T"}},
        "Mark-to-Market":  {"required": set()},
        "Relative":        {"required": set()},
        "Credit-Model":    {"required": set()},
    }
    
    param_keys = set(params.keys())
    
    # 1. Essayer le moteur recommande
    sig = ENGINE_SIGNATURES.get(method, {}).get("required", set())
    if sig.issubset(param_keys):
        engine = VALUATION_ENGINES.get(method)
        if engine:
            try:
                return engine(**params)
            except Exception as e:
                pass  # Fallback
    
    # 2. Detecter quel moteur correspond aux parametres fournis
    for candidate_method, spec in ENGINE_SIGNATURES.items():
        if spec["required"] and spec["required"].issubset(param_keys):
            engine = VALUATION_ENGINES.get(candidate_method)
            if engine:
                try:
                    result = engine(**params)
                    result["note_dispatch"] = (
                        f"Methode recommandee ({method}) incompatible avec les "
                        f"parametres fournis. Calcul effectue avec {candidate_method}."
                    )
                    return result
                except:
                    continue
    
    # 3. Aucun moteur compatible
    return {
        "method": method,
        "error": f"Parametres insuffisants pour {method}. "
                 f"Requis : {ENGINE_SIGNATURES.get(method, {}).get('required', '?')}. "
                 f"Recus : {param_keys}.",
    }


def _build_explanation_text(final_method, ml_method, confidence,
                            drivers, ifrs_override):
    """Genere le texte en langage naturel pour l'utilisateur."""
    parts = [
        f"La methode {final_method} est recommandee "
        f"avec une confiance de {confidence:.0%}."
    ]
    
    if drivers:
        top3 = ", ".join([d["feature"].replace("_", " ") for d in drivers[:3]])
        parts.append(f"Les facteurs determinants sont : {top3}.")
    
    if ifrs_override:
        parts.append(
            f"Note IFRS 13 : la prediction ML initiale ({ml_method}) "
            f"a ete corrigee vers {final_method} pour conformite reglementaire."
        )
    
    return " ".join(parts)


print("valuate_asset() V2 chargee")
print("  + _build_feature_defaults() : defaults realistes par classe d'actif")
print("  + _dispatch_valuation()     : routing intelligent des parametres")
print("  + _build_explanation_text()  : generation de texte naturel")

---
## Scénarios de Démonstration (corrigés)


In [ ]:
print("=" * 65)
print("  DEMO VALUSENSE — SCENARIOS REALISTES")
print("=" * 65)

# ──────────────────────────────────────────────────────────────
# Scenario 1 : Option call europeenne (AAPL)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 1 : Option call europeenne sur AAPL")
print("-" * 65)

r1 = valuate_asset(
    asset_features={
        # Structurelles
        "has_options_features": 1, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 0.5,
        # Enrichissement (realistes pour une option AAPL)
        "implied_volatility_atm": 0.26,
        "iv_skew": 0.04,
        "historical_vol_30d": 0.24,
        "asset_class_encoded": 4,      # Option
        "asset_subclass_encoded": 14,   # European Option
    },
    valuation_params={
        "S": 195, "K": 200, "T": 0.5, "r": 0.045,
        "sigma": 0.26, "option_type": "call"
    },
)

print(f"  Methode    : {r1['recommendation']['method']}")
print(f"  Confiance  : {r1['recommendation']['confidence']:.0%}")
v = r1['valuation']
if v and 'price' in v:
    print(f"  Prix       : {v['price']:.2f} $")
    if 'greeks' in v:
        g = v['greeks']
        print(f"  Delta={g['delta']:.3f}  Gamma={g['gamma']:.5f}  "
              f"Vega={g['vega']:.3f}  Theta={g['theta']:.4f}")
print(f"  Explication: {r1['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 2 : Obligation corporate BBB (5 ans, coupon 5%)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 2 : Obligation corporate BBB 5 ans")
print("-" * 65)

r2 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 1, "is_exchange_traded": 1,
        "has_credit_risk": 1, "volatility_available": 0,
        "liquidity": 1, "data_availability": 2,
        "ifrs_level": 2, "maturity_years": 5,
        # Enrichissement bond
        "duration_estimate": 4.3,
        "credit_spread_asset": 2.1,
        "asset_class_encoded": 0,      # Bond
        "asset_subclass_encoded": 3,    # Corporate Bond
    },
    valuation_params={
        "cash_flows": [50, 50, 50, 50, 1050],
        "discount_rate": 0.065,
        "terminal_growth": 0,
    },
)

print(f"  Methode    : {r2['recommendation']['method']}")
print(f"  Confiance  : {r2['recommendation']['confidence']:.0%}")
v = r2['valuation']
if v and 'fair_value' in v:
    print(f"  Juste val  : {v['fair_value']:.2f} $ (nominal 1000)")
    print(f"  PV flux    : {v['pv_cash_flows']:.2f}")
    print(f"  Val term   : {v['terminal_value']:.2f} ({v['terminal_pct']}%)")
print(f"  Explication: {r2['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 3 : Action a dividende (EDF — utility)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 3 : Action EDF (utility, dividende stable)")
print("-" * 65)

r3 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 1, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": -1,
        # Enrichissement equity dividend-paying
        "dividend_yield": 0.045,
        "beta": 0.65,
        "pe_ratio": 12.5,
        "market_cap": 35e9,
        "debt_to_equity": 1.8,
        "historical_vol_30d": 0.15,
        "asset_class_encoded": 2,       # Equity
        "asset_subclass_encoded": 45,    # Utility Stock
    },
    valuation_params={
        "dividend_current": 1.15,
        "growth_rate": 0.025,
        "required_return": 0.08,
    },
)

print(f"  Methode    : {r3['recommendation']['method']}")
print(f"  Confiance  : {r3['recommendation']['confidence']:.0%}")
v = r3['valuation']
if v and 'fair_value' in v:
    print(f"  Juste val  : {v['fair_value']:.2f} EUR")
    print(f"  Div yield  : {v.get('implied_dividend_yield', 'N/A')}%")
elif v and 'note_dispatch' in v:
    print(f"  Note       : {v['note_dispatch']}")
    print(f"  Juste val  : {v.get('fair_value', 'N/A')}")
print(f"  Explication: {r3['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 4 : Option americaine put (early exercise)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 4 : Option put americaine deep ITM")
print("-" * 65)

r4 = valuate_asset(
    asset_features={
        "has_options_features": 1, "has_early_exercise": 1,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 1.0,
        "implied_volatility_atm": 0.32,
        "iv_skew": 0.05,
        "historical_vol_30d": 0.30,
        "asset_class_encoded": 4,
        "asset_subclass_encoded": 1,    # American Option
    },
    valuation_params={
        "S": 80, "K": 100, "T": 1.0, "r": 0.045,
        "sigma": 0.32, "option_type": "put",
    },
)

print(f"  Methode    : {r4['recommendation']['method']}")
print(f"  Confiance  : {r4['recommendation']['confidence']:.0%}")
v = r4['valuation']
if v and 'price' in v:
    print(f"  Prix       : {v['price']:.2f} $")
    print(f"  Exercice anticipe optimal : {v.get('early_exercise_optimal', 'N/A')}")
print(f"  Explication: {r4['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 5 : Option asiatique (path-dependent)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 5 : Option asiatique (moyenne) — Monte-Carlo")
print("-" * 65)

r5 = valuate_asset(
    asset_features={
        "has_options_features": 1, "has_early_exercise": 0,
        "is_path_dependent": 1, "has_market_price": 0,
        "has_cash_flows": 0, "is_exchange_traded": 0,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 0, "data_availability": 1,
        "ifrs_level": 3, "maturity_years": 1.0,
        "implied_volatility_atm": 0.35,
        "iv_skew": 0.06,
        "historical_vol_30d": 0.33,
        "asset_class_encoded": 4,
        "asset_subclass_encoded": 7,    # Exotic Option
    },
    valuation_params={
        "S": 100, "K": 100, "T": 1.0, "r": 0.04,
        "sigma": 0.35, "exotic_type": "asian",
    },
)

print(f"  Methode    : {r5['recommendation']['method']}")
print(f"  Confiance  : {r5['recommendation']['confidence']:.0%}")
v = r5['valuation']
if v and 'price' in v:
    print(f"  Prix       : {v['price']:.4f} $")
    print(f"  IC 95%%     : {v.get('confidence_95', 'N/A')}")
    print(f"  Erreur std : {v.get('std_error', 'N/A')}")
print(f"  Explication: {r5['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 6 : Contrat forward EUR/USD
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 6 : Forward EUR/USD 3 mois")
print("-" * 65)

r6 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 0,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 0.25,
        "asset_class_encoded": 1,       # Currency
        "asset_subclass_encoded": 30,   # FX Forward
    },
    valuation_params={
        "S": 1.0850, "r_domestic": 0.045, "r_foreign": 0.035, "T": 0.25,
    },
)

print(f"  Methode    : {r6['recommendation']['method']}")
print(f"  Confiance  : {r6['recommendation']['confidence']:.0%}")
v = r6['valuation']
if v and 'forward_rate' in v:
    print(f"  Forward    : {v['forward_rate']}")
    print(f"  Fwd points : {v.get('forward_points', 'N/A')}")
print(f"  Explication: {r6['explanation']['natural_language']}")

# ──────────────────────────────────────────────────────────────
# Scenario 7 : Or (commodity)
# ──────────────────────────────────────────────────────────────
print("\n" + "-" * 65)
print("  Scenario 7 : Contrat a terme sur l'or (6 mois)")
print("-" * 65)

r7 = valuate_asset(
    asset_features={
        "has_options_features": 0, "has_early_exercise": 0,
        "is_path_dependent": 0, "has_market_price": 1,
        "has_cash_flows": 0, "is_exchange_traded": 1,
        "has_credit_risk": 0, "volatility_available": 1,
        "liquidity": 2, "data_availability": 2,
        "ifrs_level": 1, "maturity_years": 0.5,
        "convenience_yield": 0.005,
        "storage_cost_pct": 0.01,
        "asset_class_encoded": 1,       # Commodity
        "asset_subclass_encoded": 35,   # Precious Metal
    },
    valuation_params={
        "S": 2350, "r": 0.045, "T": 0.5,
        "storage_cost": 0.01, "convenience_yield": 0.005,
    },
)

print(f"  Methode    : {r7['recommendation']['method']}")
print(f"  Confiance  : {r7['recommendation']['confidence']:.0%}")
v = r7['valuation']
if v and 'forward_price' in v:
    print(f"  Forward    : {v['forward_price']:.2f} $/oz")
    print(f"  Basis      : {v.get('basis', 'N/A')}")
print(f"  Explication: {r7['explanation']['natural_language']}")

---
## Tableau Récapitulatif des Scénarios (pour le rapport)


In [ ]:
# ── Generer le tableau recapitulatif ───────────────────────────
scenarios = [
    ("Option EU call AAPL", r1),
    ("Obligation BBB 5Y", r2),
    ("Action EDF (DDM)", r3),
    ("Option US put ITM", r4),
    ("Option asiatique", r5),
    ("Forward EUR/USD", r6),
    ("Or forward 6M", r7),
]

print("=" * 85)
print(f"  {'Scenario':25s} {'Methode':18s} {'Confiance':>10s} {'Valeur':>12s} {'IFRS':>6s}")
print("-" * 85)

for name, result in scenarios:
    method = result["recommendation"]["method"]
    conf = result["recommendation"]["confidence"]
    ifrs = "Oui" if result["recommendation"]["ifrs_override"] else "Non"
    
    val = result.get("valuation", {})
    if val:
        value = (val.get("price") or val.get("fair_value") or
                 val.get("forward_price") or val.get("forward_rate") or "—")
        if isinstance(value, (int, float)):
            value = f"{value:.2f}"
    else:
        value = "—"
    
    print(f"  {name:25s} {method:18s} {conf:>9.0%} {value:>12s} {ifrs:>6s}")

print("=" * 85)